# 🚀 Notebook do Professor (Demo) — Aula 06: Pipeline RAG completo

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 06/14 — Módulo 2: RAG · load → split → embed → retrieve → generate**  
**⏱️ 1h40min**  
**📄 PyMuPDF · RecursiveCharacterTextSplitter**  
**🔁 Andaime 50%**  

---

## 🎯 Objetivo da aula

Construir um pipeline RAG completo funcional em ~50 linhas. O LLM responde sobre documentos reais que nunca estiveram no treinamento — com citação de página e trecho. Essa é a fundação do CKP02.

---

## Como usar este notebook

- Cada célula corresponde a um slide de código da aula (a ordem é a da apresentação).
- Rode ao vivo enquanto explica o slide correspondente.
- A última seção traz as soluções resolvidas dos exercícios da aula.

---

# 🔬 Código da aula — slide a slide

In [ ]:
!pip install langchain-ollama langchain-core langchain-classic -q

from langchain_ollama import ChatOllama
from google.colab import userdata
import os

# Definir a API key via variável de ambiente (Colab Secrets)
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

### Slide 08 — Etapa 1 — carregar PDFs com PyMuPDF

In [ ]:
!pip install langchain langchain-community pymupdf langchain-ollama chromadb -q

from langchain_community.document_loaders import PyMuPDFLoader
from pathlib import Path

# Fazer upload do PDF no Colab
from google.colab import files
uploaded = files.upload()  # abre o seletor de arquivos
pdf_path = list(uploaded.keys())[0]

# Carregar — cada página vira um Document
loader = PyMuPDFLoader(pdf_path)
paginas = loader.load()

print(f"Páginas carregadas: {len(paginas)}")
print(f"Metadados da pág. 1: {paginas[0].metadata}")
# → {'source': 'manual.pdf', 'page': 0, 'total_pages': 24, ...}

print(f"Trecho da pág. 1:\n{paginas[0].page_content[:300]}")

# Carregar múltiplos PDFs de uma vez
pdfs = ["manual_1.pdf", "manual_2.pdf", "regulamento.pdf"]
todas_paginas = []
for pdf in pdfs:
    todas_paginas.extend(PyMuPDFLoader(pdf).load())
print(f"Total de páginas: {len(todas_paginas)}")

### Slide 09 — Etapa 2 — dividir em chunks com RecursiveCharacterTextSplitter

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,      # ~600–1000 chars por chunk (boa prática para RAG)
    chunk_overlap=100,  # sobreposição para não quebrar frases no limite
    separators=["\n\n", "\n", ". ", " ", ""],  # ordem de preferência
)

# Dividir as páginas em chunks menores
chunks = splitter.split_documents(paginas)

print(f"Páginas originais: {len(paginas)}")
print(f"Chunks gerados:    {len(chunks)}")
print(f"Exemplo de chunk:\n{chunks[0].page_content}")
print(f"Metadados: {chunks[0].metadata}")
# → {'source': 'manual.pdf', 'page': 0} — página preservada!

# Ver distribuição de tamanhos dos chunks
tamanhos = [len(c.page_content) for c in chunks]
import statistics
print(f"Tamanho médio: {statistics.mean(tamanhos):.0f} chars")
print(f"Tamanho mín/máx: {min(tamanhos)} / {max(tamanhos)} chars")

### Slide 10 — Etapas 3 e 4 — embed e store no ChromaDB

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings
import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Etapas 3 + 4 em uma linha: embed todos os chunks e salva no ChromaDB
db = Chroma.from_documents(
    documents=chunks,                        # lista de chunks (Document)
    embedding=embeddings,                    # modelo de embedding
    persist_directory="/content/rag_ckp02",  # salva no disco
)
print(f"Chunks indexados: {db._collection.count()}")

# Retriever: interface de busca para a chain LCEL
retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},  # recuperar os 3 chunks mais similares
)

# Testar o retriever isolado
docs_recuperados = retriever.invoke("qual é o prazo de entrega?")
for d in docs_recuperados:
    pg = d.metadata.get("page", "?")
    print(f"Pág. {pg}: {d.page_content[:120]}...")

### Slide 12 — O prompt de RAG — grounding e citação de fonte

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

PROMPT_RAG = """<persona>
Você é um assistente especializado em responder perguntas
com base EXCLUSIVAMENTE nos documentos fornecidos.
</persona>

<instrucoes>
- Responda SOMENTE com informações do contexto abaixo.
- Sempre cite a fonte: (fonte: {nome_doc}, página X).
- Se a resposta não estiver no contexto, diga:
  "Não encontrei essa informação nos documentos fornecidos."
- Nunca invente, extrapole ou use conhecimento externo.
</instrucoes>

<contexto>
{contexto}
</contexto>

<pergunta>
{pergunta}
</pergunta>"""

prompt = ChatPromptTemplate.from_template(PROMPT_RAG)

### Slide 13 — Chain RAG completa — ~50 linhas, pipeline funcional

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

llm = ChatOllama(model="gpt-oss:120b", temperature=0)  # temp=0 para RAG (menos variação)

def formatar_contexto(docs) -> str:
    """Formata os docs recuperados com número de página para citação."""
    partes = []
    for i, doc in enumerate(docs, 1):
        fonte = doc.metadata.get("source", "doc")
        pg    = doc.metadata.get("page", "?")
        partes.append(f"[Trecho {i} — {fonte}, pág. {pg+1}]\n{doc.page_content}")
    return "\n\n---\n\n".join(partes)

# Chain LCEL completa — a magic pipe
chain_rag = (
    {
        "contexto":   retriever | RunnableLambda(formatar_contexto),
        "pergunta":   RunnablePassthrough(),  # pergunta passa direto
        "nome_doc":   RunnableLambda(lambda _: pdf_path),
    }
    | prompt
    | llm
    | StrOutputParser()
)

# Usar a chain
resposta = chain_rag.invoke("Qual é o prazo de entrega do produto?")
print(resposta)
# → "Conforme o documento manual.pdf, página 8, o prazo de entrega é..."

### Slide 17 — Pipeline completo em bloco único (~50 linhas)

In [ ]:
# ── 1. SETUP ────────────────────────────────────────────────
import os
from google.colab import userdata
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

# ── 2. INDEXAÇÃO (uma vez) ───────────────────────────────────
paginas  = PyMuPDFLoader("documento.pdf").load()
chunks   = RecursiveCharacterTextSplitter(800, 100).split_documents(paginas)
db       = Chroma.from_documents(chunks, OllamaEmbeddings(model="nomic-embed-text"),
                               persist_directory="/content/rag")
retriever = db.as_retriever(search_kwargs={"k":3})

# ── 3. CHAIN LCEL ───────────────────────────────────────────
def fmt(docs):
    return "\n\n---\n\n".join(
        f"[{d.metadata.get('source','?')}, pág.{d.metadata.get('page',0)+1}]\n{d.page_content}"
        for d in docs
    )

chain = (
    {"contexto": retriever | RunnableLambda(fmt),
     "pergunta": RunnablePassthrough(),
     "nome_doc": RunnableLambda(lambda _:"documento.pdf")}
    | prompt | ChatOllama("gpt-oss:120b", temperature=0) | StrOutputParser()
)

# ── 4. USAR ─────────────────────────────────────────────────
print(chain.invoke("Qual é o prazo de garantia?"))

### Slide 18 — RunnablePassthrough e RunnableLambda — as peças de cola

In [ ]:
# O que acontece quando chain.invoke("Qual é o prazo?"):
#
# Input: "Qual é o prazo?"
#   ↓
# retriever.invoke("Qual é o prazo?") → [Doc1, Doc2, Doc3]
# formatar_contexto([Doc1, Doc2, Doc3]) → "[pág.8]\n...\n---\n[pág.12]\n..."
#   ↓
# contexto = "[pág.8]\n..."       ← resultado do retriever | lambda
# pergunta = "Qual é o prazo?"    ← RunnablePassthrough() — mesma string
# nome_doc = "documento.pdf"      ← RunnableLambda(lambda _: "...")
#   ↓
# prompt.format(contexto=..., pergunta=..., nome_doc=...) → mensagens
#   ↓
# llm(mensagens) → AIMessage
#   ↓
# StrOutputParser() → string final com citação de página

### Slide 22 — Python novo desta aula

In [ ]:
# 1. extend() — adicionar todos os itens de uma lista em outra
lista_a = [1, 2]
lista_a.extend([3, 4])  # [1, 2, 3, 4] — diferente de append([3, 4]) = [1, 2, [3, 4]]

# 2. enumerate() — iterar com índice
for i, doc in enumerate(docs, 1):  # começa em 1 — i=1,2,3...
    print(f"Trecho {i}: {doc.page_content}")

# 3. .get() em dict com valor padrão
pg = doc.metadata.get("page", "?")  # retorna "?" se "page" não existir

# 4. join() com separador multilinha
separador = "\n\n---\n\n"
texto     = separador.join(["Trecho A", "Trecho B"])

# 5. RunnablePassthrough e RunnableLambda — LCEL avançado
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

passthrough = RunnablePassthrough()        # x → x (identidade)
constante   = RunnableLambda(lambda _: "doc.pdf")  # ignora input, retorna constante
transformar = RunnableLambda(minha_funcao)    # qualquer função vira Runnable

# 6. statistics.mean() — média de uma lista
import statistics
media = statistics.mean([100, 200, 150])  # 150.0

---

## 🏋️ Exercícios Resolvidos — versão professor (executar no Colab)

As quatro soluções prontas dos exercícios de fixação do notebook do aluno — rode em sala, uma a uma.


### Exercício 1 — As 5 etapas do pipeline RAG

**O que a solução demonstra:** o pipeline inteiro executado com o objeto que cada etapa produz impresso no console — `Document` por página, lista de chunks com metadata preservada, coleção indexada, top-k do retriever e a string final da chain.

**Pontos a destacar na execução:** mostre que `metadata["page"]` sobrevive ao splitter e comente o `temperature=0`. Os valores preenchidos são os da aula: chunk_size 800, overlap 100, k=3.


In [ ]:
# ── Solução do Exercício 1 — o pipeline inteiro, etapa a etapa ──
!pip install langchain langchain-community langchain-ollama pymupdf chromadb langchain-text-splitters -q

import os
from google.colab import userdata
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

from langchain_community.document_loaders import PyMuPDFLoader   # PDFs reais do grupo
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Com PDFs reais do grupo: paginas = PyMuPDFLoader("manual.pdf").load()
# Aqui, páginas de exemplo no mesmo formato que o PyMuPDF devolve:
paginas = [
    Document(page_content="Manual do produto. Garantia de 12 meses contra defeitos de fabricação. O prazo de entrega é de 5 dias úteis após a confirmação do pagamento. A garantia não cobre danos por mau uso, queda ou contato com líquidos.", metadata={"source": "manual_demo.pdf", "page": 0}),
    Document(page_content="Suporte técnico pelo portal e pelo chat, em dias úteis, das 9h às 18h, com atendimento em até 48 horas úteis após a abertura do chamado.", metadata={"source": "manual_demo.pdf", "page": 1}),
    Document(page_content="A devolução é aceita em até 7 dias corridos com a embalagem original. O reembolso ocorre em até 10 dias após o recebimento do produto devolvido.", metadata={"source": "manual_demo.pdf", "page": 2}),
]

# ETAPA 1 — LOAD: um Document por página, com metadata["page"]
print(f"① LOAD          → {len(paginas)} Documents · metadata pág. 1: {paginas[0].metadata}")

# ETAPA 2 — SPLIT: lista de chunks, metadata["page"] preservado
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(paginas)
print(f"② SPLIT         → {len(chunks)} chunks · metadata: {chunks[0].metadata}")

# ETAPA 3 — EMBED + STORE: vetores calculados e salvos no ChromaDB (Slide 10)
embeddings = OllamaEmbeddings(model="nomic-embed-text")
db = Chroma.from_documents(chunks, embeddings, collection_name="ex01",
                           persist_directory="/content/rag_ex01")
print(f"③ EMBED + STORE → {db._collection.count()} chunks indexados no ChromaDB")

# ETAPA 4 — RETRIEVE: string entra, lista de Documents sai (contrato do retriever)
retriever = db.as_retriever(search_kwargs={"k": 3})
docs_rec = retriever.invoke("qual é o prazo de entrega?")
print(f"④ RETRIEVE      → {len(docs_rec)} chunks para a pergunta")

# ETAPA 5 — GENERATE: prompt de grounding + ChatOllama(temperature=0) + parser
def formatar_contexto(docs) -> str:
    return "\n\n---\n\n".join(
        f"[Trecho {i} — {d.metadata.get('source','?')}, pág. {d.metadata.get('page',0)+1}]\n{d.page_content}"
        for i, d in enumerate(docs, 1)
    )

PROMPT_RAG = """<persona>
Você é um assistente especializado em responder perguntas
com base EXCLUSIVAMENTE nos documentos fornecidos.
</persona>

<instrucoes>
- Responda SOMENTE com informações do contexto abaixo.
- Sempre cite a fonte: (fonte: {nome_doc}, página X).
- Se a resposta não estiver no contexto, diga:
  "Não encontrei essa informação nos documentos fornecidos."
- Nunca invente, extrapole ou use conhecimento externo.
</instrucoes>

<contexto>
{contexto}
</contexto>

<pergunta>
{pergunta}
</pergunta>"""
prompt = ChatPromptTemplate.from_template(PROMPT_RAG)

chain_rag = (
    {"contexto": retriever | RunnableLambda(formatar_contexto),
     "pergunta": RunnablePassthrough(),
     "nome_doc": RunnableLambda(lambda _: "manual_demo.pdf")}
    | prompt | ChatOllama(model="gpt-oss:120b", temperature=0) | StrOutputParser()
)
print("⑤ GENERATE      →")
print(chain_rag.invoke("Qual é o prazo de entrega do produto?"))
# → resposta com citação (fonte: manual_demo.pdf, página X)


### Exercício 2 — Prompt que perde o grounding

**O que a solução demonstra:** o prompt "vazado" e o corrigido em execução — as lacunas do andaime preenchidas (o que responde, a página citada e o fallback) e a prova com pergunta fora dos documentos.

**Pontos a destacar na execução:** a chain corrigida responde "Não encontrei essa informação nos documentos fornecidos." em vez de chutar — esse grounding estrito é o que a RAGAS da Aula 07 medirá como `faithfulness`.


In [ ]:
# ── Solução do Exercício 2 (parte 1) — os dois prompts, lado a lado ──
!pip install langchain langchain-community langchain-ollama chromadb langchain-text-splitters -q

import os
from google.colab import userdata
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Base mínima para a prova — com PDFs reais: paginas = PyMuPDFLoader(pdf).load()
paginas = [
    Document(page_content="Manual do produto. Garantia de 12 meses contra defeitos de fabricação. Entrega em 5 dias úteis após a confirmação do pagamento.", metadata={"source": "manual_demo.pdf", "page": 0}),
    Document(page_content="Suporte técnico pelo portal e pelo chat, em dias úteis, das 9h às 18h, com atendimento em até 48 horas úteis.", metadata={"source": "manual_demo.pdf", "page": 1}),
]
db = Chroma.from_documents(
    RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100).split_documents(paginas),
    embeddings, collection_name="ex02_grounding",
)
retriever = db.as_retriever(search_kwargs={"k": 3})

def formatar_contexto(docs) -> str:
    return "\n\n---\n\n".join(
        f"[{d.metadata.get('source','?')}, pág. {d.metadata.get('page',0)+1}]\n{d.page_content}"
        for d in docs
    )

# Versão VAZADA — autoriza conhecimento geral (perde o grounding)
PROMPT_VAZADO = """Você é um assistente prestativo.
Responda a pergunta usando o contexto e, se precisar,
use também o seu conhecimento geral.

<contexto>{contexto}</contexto>
<pergunta>{pergunta}</pergunta>"""
# Riscos: (1) responde com informação que não está nos PDFs (alucinação);
#         (2) cita páginas que não existem para dar credibilidade.

# Versão CORRIGIDA — grounding estrito (idêntico ao Slide 12), lacunas preenchidas
PROMPT_RAG = """<persona>
Você é um assistente especializado em responder perguntas
com base EXCLUSIVAMENTE nos documentos fornecidos.
</persona>

<instrucoes>
- Responda SOMENTE com informações do contexto abaixo.
- Sempre cite a fonte: (fonte: {nome_doc}, página X).
- Se a resposta não estiver no contexto, diga:
  "Não encontrei essa informação nos documentos fornecidos."
- Nunca invente, extrapole ou use conhecimento externo.
</instrucoes>

<contexto>
{contexto}
</contexto>

<pergunta>
{pergunta}
</pergunta>"""
prompt_vazado    = ChatPromptTemplate.from_template(PROMPT_VAZADO)
prompt_corrigido = ChatPromptTemplate.from_template(PROMPT_RAG)


In [ ]:
# ── Solução do Exercício 2 (parte 2) — a prova do grounding ──
chain_vazada = (
    {"contexto": retriever | RunnableLambda(formatar_contexto),
     "pergunta": RunnablePassthrough()}
    | prompt_vazado | ChatOllama(model="gpt-oss:120b", temperature=0) | StrOutputParser()
)
chain_prova = (
    {"contexto": retriever | RunnableLambda(formatar_contexto),
     "pergunta": RunnablePassthrough(),
     "nome_doc": RunnableLambda(lambda _: "manual_demo.pdf")}
    | prompt_corrigido | ChatOllama(model="gpt-oss:120b", temperature=0) | StrOutputParser()
)

pergunta_fora = "Qual é o preço do ingresso do cinema?"   # NÃO está nos documentos
print("VERSÃO VAZADA (risco de alucinação):")
print(chain_vazada.invoke(pergunta_fora))
print("\nVERSÃO CORRIGIDA (grounding estrito):")
print(chain_prova.invoke(pergunta_fora))
# Esperado da corrigida: "Não encontrei essa informação nos documentos fornecidos."


### Exercício 3 — Carregamento de PDF com PyMuPDF

**O que a solução demonstra:** o caminho completo do load — PDF gerado em disco, `PyMuPDFLoader` devolvendo um `Document` por página com `metadata["page"]` preservado e o trecho impresso.

**Pontos a destacar na execução:** em sala, troque o PDF gerado pelos PDFs do grupo (o upload do andaime) — o restante do código não muda. Destaque o contrato: 1 página = 1 Document com a página preservada.


In [ ]:
# ── Solução do Exercício 3 — carregamento de PDF com PyMuPDF ──
!pip install langchain langchain-community pymupdf -q

import fitz   # PyMuPDF

# Gera um PDF de demonstração — em sala, troque pelos PDFs do grupo via files.upload()
doc_pdf = fitz.open()
pagina = doc_pdf.new_page()
pagina.insert_text((72, 72),  "Manual do produto - Garantia de 12 meses contra defeitos de fabricação.")
pagina.insert_text((72, 92),  "O prazo de entrega é de 5 dias úteis após a confirmação do pagamento.")
pagina.insert_text((72, 112), "Suporte técnico pelo portal e pelo chat, em dias úteis, das 9h às 18h.")
doc_pdf.save("/content/manual_demo.pdf")
doc_pdf.close()
print("PDF de demonstração criado: /content/manual_demo.pdf")

from langchain_community.document_loaders import PyMuPDFLoader

# Lacuna 1 resolvida — o loader recebe o caminho do arquivo
loader = PyMuPDFLoader("/content/manual_demo.pdf")

# Lacuna 2 resolvida — load() devolve um Document por página
paginas_pdf = loader.load()
print(f"Páginas carregadas: {len(paginas_pdf)}")

# Lacuna 3 resolvida — a chave "page" do metadata guarda o número da página
print(f"Metadados da pág. 1: {paginas_pdf[0].metadata}")
print(f"Trecho da pág. 1:\n{paginas_pdf[0].page_content[:300]}")

# Vários PDFs de uma vez (com o upload do grupo: pdf_paths = list(uploaded.keys()))
todas = []
for p in ["/content/manual_demo.pdf"]:
    todas.extend(PyMuPDFLoader(p).load())
print(f"Total de páginas: {len(todas)}")


### Exercício 4 — Prova anti-alucinação

**O que a solução demonstra:** o teste de grounding à prova — duas perguntas com resposta nos documentos e uma claramente fora deles, todas rodando na chain com grounding estrito.

**Pontos a destacar na execução:** as duas primeiras devem voltar com citação `(fonte, página X)`; a terceira deve retornar o fallback "Não encontrei essa informação nos documentos fornecidos." Se alucinar, confirme `temperature=0` e as instruções do Slide 12 e rode novamente.


In [ ]:
# ── Solução do Exercício 4 (parte 1) — chain RAG com grounding estrito ──
!pip install langchain langchain-community langchain-ollama chromadb langchain-text-splitters -q

import os
from google.colab import userdata
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

embeddings = OllamaEmbeddings(model="nomic-embed-text")

paginas = [
    Document(page_content="Manual do produto. Garantia de 12 meses contra defeitos de fabricação. O prazo de entrega é de 5 dias úteis após a confirmação do pagamento.", metadata={"source": "manual_demo.pdf", "page": 0}),
    Document(page_content="Suporte técnico pelo portal e pelo chat, em dias úteis, das 9h às 18h, com atendimento em até 48 horas úteis.", metadata={"source": "manual_demo.pdf", "page": 1}),
]
db = Chroma.from_documents(
    RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100).split_documents(paginas),
    embeddings, collection_name="ex04_prova",
)
retriever = db.as_retriever(search_kwargs={"k": 3})

def formatar_contexto(docs) -> str:
    return "\n\n---\n\n".join(
        f"[{d.metadata.get('source','?')}, pág. {d.metadata.get('page',0)+1}]\n{d.page_content}"
        for d in docs
    )

PROMPT_RAG = """<persona>
Você é um assistente especializado em responder perguntas
com base EXCLUSIVAMENTE nos documentos fornecidos.
</persona>

<instrucoes>
- Responda SOMENTE com informações do contexto abaixo.
- Sempre cite a fonte: (fonte: {nome_doc}, página X).
- Se a resposta não estiver no contexto, diga:
  "Não encontrei essa informação nos documentos fornecidos."
- Nunca invente, extrapole ou use conhecimento externo.
</instrucoes>

<contexto>
{contexto}
</contexto>

<pergunta>
{pergunta}
</pergunta>"""

chain_prova = (
    {"contexto": retriever | RunnableLambda(formatar_contexto),
     "pergunta": RunnablePassthrough(),
     "nome_doc": RunnableLambda(lambda _: "manual_demo.pdf")}
    | ChatPromptTemplate.from_template(PROMPT_RAG)
    | ChatOllama(model="gpt-oss:120b", temperature=0) | StrOutputParser()
)
print("Chain com grounding estrito pronta para a prova.")


In [ ]:
# ── Solução do Exercício 4 (parte 2) — a prova anti-alucinação ──
perguntas_prova = [
    "Qual é o prazo de garantia do produto?",     # ESTÁ nos documentos
    "Como acionar o suporte técnico?",            # ESTÁ nos documentos
    "Qual o nome do time de futebol do autor?",   # NÃO está nos documentos
]
for q in perguntas_prova:
    print(f"\n📌 {q}")
    print(chain_prova.invoke(q))
# Esperado: as 2 primeiras com citação (fonte, página X);
# a 3ª retorna "Não encontrei essa informação nos documentos fornecidos."
# Se a 3ª alucinar: confirme temperature=0 e as instruções do Slide 12 —
# "Nunca invente, extrapole ou use conhecimento externo." — e rode novamente.


## 📚 Referências da aula

- Paper Lewis, P. et al. — "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks." NeurIPS, 2020. O paper original que cunhou o termo RAG. arxiv.org/abs/2005.11401
- Docs LangChain — RAG tutorial completo com PyMuPDF, Chroma e LCEL. python.langchain.com/docs/tutorials/rag
- Docs PyMuPDF — Documentação do loader LangChain com PyMuPDF. python.langchain.com/docs/integrations/document_loaders/pymupdf
- Docs RecursiveCharacterTextSplitter — Estratégias de chunking, parâmetros e separadores. python.langchain.com/docs/how_to/recursive_text_splitter
- Livro Goodfellow, I.; Bengio, Y.; Courville, A. — Deep Learning. Pearson, 2017. Cap. 15 — Representações distribuídas: a base teórica dos embeddings usados no RAG.

---

**→ Próxima Aula — Aula 07 · 21/09** — RAG avançado — chunking estratégico, reranking e RAGAS
  
Medir faithfulness e answer relevancy. Otimizar o pipeline. Entregar CKP02.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*